## W tym pliku przygotujemy dane pod trenowanie modelu uczenia maszynowego


In [10]:
#Sprytny sposób na wczytanie plików, wczytujemy dane dla 2024 roku z citybikes
import pandas as pd
import os
import glob
path = "C:\\Dane\\Projekt_PDU"
all_files = glob.glob(os.path.join(path,"2024*.csv"))
lista = []
wybrane_kolumny = ["started_at","ended_at"]
for file in all_files:
    lista.append(pd.read_csv(file,usecols = wybrane_kolumny))
df = pd.concat(lista,axis = 0,ignore_index=True)
print(df.shape)
print(df.head)

KeyboardInterrupt: 

In [2]:
#Konswersja typów na daty
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])
df["Day"] = df["started_at"].dt.date

In [3]:
#Grupowanie po dacie
df_unique = df.groupby(["Day"]).size().reset_index(name = "Liczba_wypozyczen")
df_unique.rename(columns = {"Day":"DATE"},inplace = True)

### WCZYTANIE DANYCH Z NOAA
Zostawimy w niej interesujące nas dane czyli

- DATE : data
- PRCP : Dobowa suma opadów
- SNOW : Dobowa suma opadów śniegu
- TMAX : Dobowa maksymalna temperatura
- TMIN : Dobowa minimalna temperatura
- AWND : Średnia prędkość wiatru

In [4]:
df_weather = pd.read_csv("C:\\Dane\\Projekt_PDU\\Dane_pogodowe.csv")

In [5]:
print(df_weather.columns)
df_weather_to_merge = df_weather[["DATE","PRCP","SNOW","AWND","TMAX","TMIN"]]
print(df_weather_to_merge.tail(20))
print(df_weather_to_merge["TMAX"].isna().sum())

Index(['STATION', 'NAME', 'DATE', 'AWND', 'PGTM', 'PRCP', 'SNOW', 'SNWD',
       'TAVG', 'TMAX', 'TMIN', 'WDF2', 'WDF5', 'WSF2', 'WSF5'],
      dtype='str')
           DATE  PRCP  SNOW   AWND  TMAX  TMIN
346  2024-12-12  0.00   0.0   9.40    41    29
347  2024-12-13  0.00   0.0   6.71    33    27
348  2024-12-14  0.00   0.0   4.92    34    26
349  2024-12-15  0.06   0.0   5.14    41    28
350  2024-12-16  0.91   0.0   5.14    52    38
351  2024-12-17  0.02   0.0   5.14    59    49
352  2024-12-18  0.32   0.0   2.68    53    42
353  2024-12-19  0.05   0.0   6.71    45    37
354  2024-12-20  0.04   0.0   8.50    37    33
355  2024-12-21  0.15   1.8  10.51    33    19
356  2024-12-22  0.00   0.0   7.61    21    13
357  2024-12-23  0.00   0.0   3.13    31    13
358  2024-12-24  0.10   1.0   4.92    39    28
359  2024-12-25  0.00   0.0   3.80    35    28
360  2024-12-26  0.00   0.0   3.13    36    24
361  2024-12-27  0.00   0.0   1.12    44    26
362  2024-12-28  0.56   0.0   1.34    54    

## Teraz zmergujemy obie te bazy : )

In [18]:
df_weather_to_merge["DATE"] = pd.to_datetime(df_weather_to_merge["DATE"])
df_unique["DATE"] = pd.to_datetime(df_unique["DATE"])
df = pd.merge(df_unique,df_weather_to_merge,on = "DATE",how = "inner")
print(df.tail())

          DATE  Liczba_wypozyczen  PRCP  SNOW  AWND  TMAX  TMIN
333 2024-12-27              54104  0.00   0.0  1.12    44    26
334 2024-12-28              26719  0.56   0.0  1.34    54    40
335 2024-12-29              61454  0.16   0.0  4.70    60    50
336 2024-12-30              82948  0.37   0.0  6.49    58    49
337 2024-12-31              76743  0.40   0.0  5.14    53    44


In [20]:
#Zawracamy uwagę na weekendy i dni wolne
weekendy_2024 = [
    '2024-01-06', '2024-01-07', '2024-01-13', '2024-01-14', '2024-01-20', '2024-01-21', '2024-01-27', '2024-01-28',
    '2024-02-03', '2024-02-04', '2024-02-10', '2024-02-11', '2024-02-17', '2024-02-18', '2024-02-24', '2024-02-25',
    '2024-03-02', '2024-03-03', '2024-03-09', '2024-03-10', '2024-03-16', '2024-03-17', '2024-03-23', '2024-03-24', '2024-03-30', '2024-03-31',
    '2024-04-06', '2024-04-07', '2024-04-13', '2024-04-14', '2024-04-20', '2024-04-21', '2024-04-27', '2024-04-28',
    '2024-05-04', '2024-05-05', '2024-05-11', '2024-05-12', '2024-05-18', '2024-05-19', '2024-05-25', '2024-05-26',
    '2024-06-01', '2024-06-02', '2024-06-08', '2024-06-09', '2024-06-15', '2024-06-16', '2024-06-22', '2024-06-23', '2024-06-29', '2024-06-30',
    '2024-07-06', '2024-07-07', '2024-07-13', '2024-07-14', '2024-07-20', '2024-07-21', '2024-07-27', '2024-07-28',
    '2024-08-03', '2024-08-04', '2024-08-10', '2024-08-11', '2024-08-17', '2024-08-18', '2024-08-24', '2024-08-25', '2024-08-31',
    '2024-09-01', '2024-09-07', '2024-09-08', '2024-09-14', '2024-09-15', '2024-09-21', '2024-09-22', '2024-09-28', '2024-09-29',
    '2024-10-05', '2024-10-06', '2024-10-12', '2024-10-13', '2024-10-19', '2024-10-20', '2024-10-26', '2024-10-27',
    '2024-11-02', '2024-11-03', '2024-11-09', '2024-11-10', '2024-11-16', '2024-11-17', '2024-11-23', '2024-11-24', '2024-11-30',
    '2024-12-01', '2024-12-07', '2024-12-08', '2024-12-14', '2024-12-15', '2024-12-21', '2024-12-22', '2024-12-28', '2024-12-29'
]
dni_wolne_2024 = [
    '2024-01-01',  # Nowy Rok
    '2024-01-06',  # Święto Trzech Króli
    '2024-04-01',  # Poniedziałek Wielkanocny (Wielkanoc 31.03 to niedziela)
    '2024-05-01',  # Święto Pracy
    '2024-05-03',  # Święto Konstytucji 3 Maja
    '2024-05-30',  # Boże Ciało
    '2024-08-15',  # Wniebowzięcie NMP / Święto Wojska Polskiego
    '2024-11-01',  # Wszystkich Świętych
    '2024-11-11',  # Narodowe Święto Niepodległości
    '2024-12-25',  # Pierwszy dzień Bożego Narodzenia
    '2024-12-26'   # Drugi dzień Bożego Narodzenia
]
free_work_date = pd.to_datetime(dni_wolne_2024).astype("datetime64[s]")
weekends_date = pd.to_datetime(weekendy_2024).astype('datetime64[s]')
df["weekend"] = df["DATE"].isin(weekends_date)
df["free_from_work"] = df["DATE"].isin(free_work_date)
print(df.head(20))

         DATE  Liczba_wypozyczen  PRCP  SNOW   AWND  TMAX  TMIN  weekend  \
0  2024-01-01              47278  0.03   0.0   3.36    47    35    False   
1  2024-01-02              68149  0.00   0.0   4.03    42    29    False   
2  2024-01-03              76559  0.00   0.0   4.92    43    34    False   
3  2024-01-04              77481  0.00   0.0   7.61    45    28    False   
4  2024-01-05              72462  0.00   0.0   7.16    37    26    False   
5  2024-01-06              43303  0.41   0.2   6.93    38    31     True   
6  2024-01-07              35678  0.24   0.0   7.61    38    34     True   
7  2024-01-08              78486  0.00   0.0   5.14    45    36    False   
8  2024-01-09              43043  1.73   0.0   6.49    57    36    False   
9  2024-01-10              81677  0.22   0.0   8.50    57    44    False   
10 2024-01-11              89191  0.00   0.0   6.93    47    41    False   
11 2024-01-12              86113  0.08   0.0   6.93    50    40    False   
12 2024-01-1

## Zapis danych dla naszego RandomForrestRegressor

In [22]:
df.rename(columns = {"PRCP" : "rain","SNOW" : "snow","DATE" : "date","AWND" : "wind"},inplace = True)
df.to_csv("C:\\Users\\kacpe\\OneDrive\\Dokumenty\\APolitechnika\\Projekty\\Projekt_koncowy_PDU\\dane_do_sieci.csv",index = False)